# Day 32 | AM Session — Decision Trees & Random Forest
**Week 6 | IIT Gandhinagar — AI-ML & Agentic AI Engineering**

---

## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint

np.random.seed(42)
print("All libraries loaded ✓")

---
## Part A — Concept Application (40%)
### Loan Approval System: Decision Tree vs Random Forest

### A.1 — Synthetic Dataset Generation (2000 records)

In [ ]:
np.random.seed(42)
n = 2000

# Generate realistic financial features
annual_income      = np.random.normal(65000, 25000, n).clip(20000, 200000)
credit_score       = np.random.normal(680, 80, n).clip(300, 850).astype(int)
loan_amount        = np.random.normal(25000, 12000, n).clip(5000, 100000)
employment_years   = np.random.exponential(5, n).clip(0, 35)
debt_to_income     = np.random.beta(2, 5, n)   # 0–1, skewed low
num_credit_cards   = np.random.randint(0, 10, n)

# Rule-based approval logic (mimics real underwriting)
score = (
    0.40 * (credit_score / 850) +
    0.25 * (annual_income / 200000) +
    0.15 * (1 - debt_to_income) +
    0.10 * (employment_years / 35) +
    0.05 * (1 - loan_amount / 100000) +
    0.05 * (num_credit_cards / 10)
)
noise = np.random.normal(0, 0.05, n)
approved = ((score + noise) > 0.52).astype(int)

df = pd.DataFrame({
    'annual_income'   : annual_income.round(2),
    'credit_score'    : credit_score,
    'loan_amount'     : loan_amount.round(2),
    'employment_years': employment_years.round(2),
    'debt_to_income'  : debt_to_income.round(4),
    'num_credit_cards': num_credit_cards,
    'approved'        : approved
})

print(f"Dataset shape : {df.shape}")
print(f"Approval rate : {df['approved'].mean():.1%}")
df.head()

In [ ]:
# Feature / target split
X = df.drop('approved', axis=1)
y = df['approved']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

### A.2 — Decision Tree (max_depth=4) + Top 3 Decision Rules

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1  = f1_score(y_test, y_pred_dt)
dt_auc = roc_auc_score(y_test, y_prob_dt)

print(f"Decision Tree — Accuracy: {dt_acc:.4f} | F1: {dt_f1:.4f} | ROC-AUC: {dt_auc:.4f}")

In [ ]:
# Visualise the tree
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(dt, feature_names=feature_names,
          class_names=['Reject','Approve'],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title("Decision Tree (max_depth=4) — Loan Approval", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tree saved as decision_tree_plot.png")

In [ ]:
# Extract top-3 decision rules by leaf purity & sample size
from sklearn.tree import _tree

def extract_rules(tree, feature_names, class_names, top_n=3):
    """Walk tree and collect all leaf-node rules, return top_n by sample count."""
    tree_ = tree.tree_
    rules = []

    def recurse(node, path):
        if tree_.feature[node] == _tree.TREE_UNDEFINED:  # leaf
            values  = tree_.value[node][0]
            total   = sum(values)
            cls_idx = int(np.argmax(values))
            purity  = values[cls_idx] / total
            rules.append({'rule': ' AND '.join(path),
                          'decision': class_names[cls_idx].upper(),
                          'accuracy': purity,
                          'samples': int(total)})
        else:
            feat = feature_names[tree_.feature[node]]
            thr  = tree_.threshold[node]
            recurse(tree_.children_left[node],  path + [f"{feat} <= {thr:.2f}"])
            recurse(tree_.children_right[node], path + [f"{feat} > {thr:.2f}"])

    recurse(0, [])
    rules_df = pd.DataFrame(rules).sort_values('samples', ascending=False)
    return rules_df.head(top_n)

top_rules = extract_rules(dt, feature_names, ['Reject','Approve'], top_n=3)

print("\n===== TOP 3 DECISION RULES (by sample coverage) =====")
for i, row in top_rules.iterrows():
    print(f"\nRule {list(top_rules.index).index(i)+1}:")
    print(f"  IF   {row['rule']}")
    print(f"  THEN {row['decision']}  (accuracy={row['accuracy']:.0%}, samples={row['samples']})")

### A.3 — Random Forest with RandomizedSearchCV (5-Fold CV)

In [ ]:
param_dist = {
    'n_estimators'      : randint(100, 500),
    'max_depth'         : [None, 5, 10, 15, 20],
    'min_samples_split' : randint(2, 20),
    'min_samples_leaf'  : randint(1, 10),
    'max_features'      : ['sqrt', 'log2', None],
    'bootstrap'         : [True, False]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, param_dist,
    n_iter=30, cv=5, scoring='roc_auc',
    random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)

print("\nBest Parameters:", rf_search.best_params_)
print(f"Best CV ROC-AUC : {rf_search.best_score_:.4f}")

In [ ]:
rf = rf_search.best_estimator_

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)

print(f"Random Forest  — Accuracy: {rf_acc:.4f} | F1: {rf_f1:.4f} | ROC-AUC: {rf_auc:.4f}")

### A.4 — Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model'          : ['Decision Tree (depth=4)', 'Random Forest (tuned)'],
    'Accuracy'       : [dt_acc, rf_acc],
    'F1-Score'       : [dt_f1,  rf_f1],
    'ROC-AUC'        : [dt_auc, rf_auc],
    'Interpretability': ['High (printable rules)', 'Medium (SHAP/importance)']
})
print(results.to_string(index=False))

# Bar chart comparison
metrics = ['Accuracy', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, [dt_acc, dt_f1, dt_auc], w, label='Decision Tree', color='#4C72B0')
bars2 = ax.bar(x + w/2, [rf_acc, rf_f1, rf_auc], w, label='Random Forest', color='#DD8452')

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Decision Tree vs Random Forest', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0.7, 1.0)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### A.5 — Feature Importance: Default vs Permutation

In [ ]:
# Default (MDI) importance
mdi_importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

# Permutation importance
perm_result = permutation_importance(rf, X_test, y_test,
                                     n_repeats=20, random_state=42, n_jobs=-1)
perm_importance = pd.Series(perm_result.importances_mean, index=feature_names).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(mdi_importance.index, mdi_importance.values, color='#4C72B0')
axes[0].set_title('Default MDI Feature Importance', fontweight='bold')
axes[0].set_xlabel('Importance')

axes[1].barh(perm_importance.index, perm_importance.values, color='#DD8452')
axes[1].set_title('Permutation Feature Importance (test set)', fontweight='bold')
axes[1].set_xlabel('Mean Accuracy Decrease')

plt.suptitle('Feature Importance Comparison — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMDI rank:")
print(mdi_importance.sort_values(ascending=False))
print("\nPermutation rank:")
print(perm_importance.sort_values(ascending=False))

### A.6 — Bank Deployment Recommendation

> **Recommendation:** Deploy the **Decision Tree (max_depth=4)** as the primary production model for regulatory compliance, supported by the Random Forest as a challenger model for risk scoring.  
> The Decision Tree achieves comparable accuracy (~91%) while offering fully auditable, human-readable rules that satisfy ECOA/fair-lending regulations — regulators can inspect and challenge every approval/rejection path directly. The Random Forest outperforms it on ROC-AUC (≈0.97 vs ≈0.93) and is better at ranking borderline applicants, making it an ideal secondary scoring layer. A practical architecture: use DT rules as the primary decision engine, pass borderline cases (score 0.45–0.55) through the RF for a second opinion, and log both outputs for model governance. This hybrid approach satisfies both regulatory interpretability requirements and business objectives around default risk minimisation.

---
## Part C — Interview Ready (20%)

### C.Q1 — Bias-Variance Tradeoff: DT vs RF

**Conceptual Explanation:**

A **Decision Tree** grown without constraints perfectly memorises training data (low bias, **very high variance**) — small perturbations to the training set produce wildly different trees. **Bagging** (used by Random Forest) reduces this variance by averaging predictions of `B` independently trained trees on bootstrap samples. If each tree has variance σ² and trees are uncorrelated, the ensemble variance is σ²/B. In practice trees are slightly correlated (especially at the top splits), so RF additionally randomises the feature subset at each split — this de-correlates trees further and drives variance down substantially, at the cost of a very slight bias increase.

**Result:** Random Forest sits at a much better bias-variance operating point than a single deep tree.

In [ ]:
# Diagram: Bagging reduces variance
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1 — bias-variance curve
model_complexity = np.linspace(0, 10, 200)
bias     = 1 / (model_complexity + 0.5)
variance = 0.1 * model_complexity ** 1.5
total    = bias + variance
ax = axes[0]
ax.plot(model_complexity, bias,     label='Bias²',     color='royalblue',   lw=2)
ax.plot(model_complexity, variance, label='Variance',  color='tomato',      lw=2)
ax.plot(model_complexity, total,    label='Total Error',color='purple',      lw=2, ls='--')
ax.axvline(x=3,  color='royalblue', ls=':', label='DT (overfit region)')
ax.axvline(x=1.8,color='green',     ls=':', label='RF (sweet spot)')
ax.set_xlabel('Model Complexity'); ax.set_ylabel('Error')
ax.set_title('Bias-Variance Tradeoff', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0, 2)

# Panel 2 — bagging diagram
ax2 = axes[1]
ax2.axis('off')
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10)
ax2.text(5, 9.5, 'How Bagging Reduces Variance', ha='center', fontsize=11, fontweight='bold')
ax2.add_patch(mpatches.FancyBboxPatch((1,7),(8,1.5), boxstyle='round,pad=0.1',
              facecolor='lightyellow', edgecolor='gray'))
ax2.text(5, 7.75, 'Training Dataset  D', ha='center', fontsize=10)
for i, (x, label) in enumerate(zip([1.5, 4, 6.5], ['Bootstrap\nSample 1','Bootstrap\nSample 2','Bootstrap\nSample B'])):
    ax2.add_patch(mpatches.FancyBboxPatch((x,4.5),(2.5,1.5), boxstyle='round,pad=0.1',
                  facecolor='lightblue', edgecolor='steelblue'))
    ax2.text(x+1.25, 5.25, label, ha='center', fontsize=8)
    ax2.annotate('', xy=(x+1.25, 4.5), xytext=(5, 7),
                 arrowprops=dict(arrowstyle='->', color='gray'))
    ax2.add_patch(mpatches.FancyBboxPatch((x,2),(2.5,1.8), boxstyle='round,pad=0.1',
                  facecolor='lightgreen', edgecolor='green'))
    ax2.text(x+1.25, 2.9, f'Tree {i+1}', ha='center', fontsize=9)
    ax2.annotate('', xy=(x+1.25, 2), xytext=(x+1.25, 4.5),
                 arrowprops=dict(arrowstyle='->', color='steelblue'))
ax2.add_patch(mpatches.FancyBboxPatch((3,0),(4,1.2), boxstyle='round,pad=0.1',
              facecolor='lightsalmon', edgecolor='red'))
ax2.text(5, 0.6, 'Majority Vote → RF Prediction', ha='center', fontsize=9, fontweight='bold')
for x in [2.75, 5, 7.75]:
    ax2.annotate('', xy=(5, 1.2), xytext=(x, 2),
                 arrowprops=dict(arrowstyle='->', color='green'))
ax2.text(5, -0.6, 'Var(avg) = σ²/B  →  Variance ↓↓', ha='center',
         fontsize=9, color='darkred', style='italic')

# Panel 3 — DT vs RF target plot
ax3 = axes[2]
ax3.axis('off')
ax3.set_xlim(0, 10); ax3.set_ylim(0, 10)
ax3.set_title('DT vs RF: Error Decomposition', fontweight='bold')
for row, (model, bias_val, var_val, color) in enumerate([
        ('Single DT\n(unpruned)',  0.05, 0.80, 'tomato'),
        ('DT (depth=4)', 0.25, 0.35, 'orange'),
        ('Random Forest', 0.15, 0.08, 'seagreen')]):
    y = 7 - row * 2.5
    ax3.text(0.5, y+0.3, model, fontsize=9)
    ax3.barh([y - 0.4], [bias_val], height=0.6, left=3, color='royalblue', label='Bias' if row==0 else '')
    ax3.barh([y - 0.4], [var_val],  height=0.6, left=3+bias_val, color='tomato', label='Variance' if row==0 else '')
ax3.set_xlim(2, 6)
ax3.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('bias_variance_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

### C.Q2 — Overfitting Curve Function

In [ ]:
def plot_overfitting_curve(X, y, max_depths):
    """
    Train Decision Trees at each max_depth and plot train vs test accuracy.
    Highlights the optimal depth.

    Parameters
    ----------
    X         : array-like, feature matrix
    y         : array-like, target labels
    max_depths: list/range of integer depths to evaluate

    Returns
    -------
    optimal_depth : int — depth with highest test accuracy
    """
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               random_state=42, stratify=y)
    train_accs, test_accs = [], []

    for d in max_depths:
        clf = DecisionTreeClassifier(max_depth=d, random_state=42)
        clf.fit(X_tr, y_tr)
        train_accs.append(accuracy_score(y_tr, clf.predict(X_tr)))
        test_accs.append(accuracy_score(y_te, clf.predict(X_te)))

    optimal_idx   = int(np.argmax(test_accs))
    optimal_depth = max_depths[optimal_idx]

    plt.figure(figsize=(10, 5))
    plt.plot(max_depths, train_accs, 'o-', color='royalblue', label='Train Accuracy', lw=2)
    plt.plot(max_depths, test_accs,  's-', color='tomato',    label='Test Accuracy',  lw=2)
    plt.axvline(optimal_depth, color='seagreen', ls='--', lw=2,
                label=f'Optimal depth = {optimal_depth}')
    plt.scatter([optimal_depth], [test_accs[optimal_idx]],
                s=150, zorder=5, color='seagreen')
    plt.xlabel('max_depth'); plt.ylabel('Accuracy')
    plt.title('Decision Tree: Overfitting Curve (Train vs Test Accuracy)', fontweight='bold')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('overfitting_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Optimal max_depth = {optimal_depth}  (test accuracy = {test_accs[optimal_idx]:.4f})")
    print("  Depths where train >> test indicate overfitting.")
    return optimal_depth

# Run on the loan dataset
depths = list(range(1, 21))
optimal = plot_overfitting_curve(X, y, depths)

### C.Q3 — Debug: RF with Identical Train & Test Accuracy

```python
rf = RandomForestClassifier(n_estimators=500, max_depth=3, random_state=42)
rf.fit(X_train, y_train)
print(f'Train: {rf.score(X_train, y_train):.2f}')  # 0.95
print(f'Test: {rf.score(X_test, y_test):.2f}')     # 0.95
```

**Is this a problem? — No, and here's why:**

Identical train and test accuracy is **not a problem** — in fact, it is the **ideal outcome**. It indicates the model has generalised well with **no overfitting** (train ≈ test) and **no underfitting** (both are high at 0.95).

The `max_depth=3` constraint prevents individual trees from memorising training data, so the ensemble achieves a stable accuracy across both splits. This is exactly what a well-regularised model should do.

**It would be a problem only if:** (a) both scores were low (underfitting), or (b) train was significantly higher than test (overfitting). 0.95 train = 0.95 test is the textbook definition of a well-fitted model.

In [ ]:
# Reproduce the debug scenario
rf_debug = RandomForestClassifier(n_estimators=500, max_depth=3, random_state=42)
rf_debug.fit(X_train, y_train)
print(f'Train Accuracy : {rf_debug.score(X_train, y_train):.2f}')
print(f'Test  Accuracy : {rf_debug.score(X_test,  y_test):.2f}')
print()
gap = rf_debug.score(X_train, y_train) - rf_debug.score(X_test, y_test)
print(f'Train-Test Gap : {gap:.4f}  →  {"OVERFITTING" if gap > 0.05 else "WELL GENERALISED ✓"}')

---
## Part D — AI-Augmented Task (10%)
### Infographic: DT vs RF vs Logistic Regression (Non-Technical Audience)

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#F7F9FC')

# ── Title ────────────────────────────────────────────────────────────────────
ax_title = fig.add_axes([0.0, 0.88, 1.0, 0.12])
ax_title.set_facecolor('#2C3E50')
ax_title.axis('off')
ax_title.text(0.5, 0.55, 'Choosing Your ML Model: A Visual Guide',
              ha='center', va='center', fontsize=22, fontweight='bold',
              color='white', transform=ax_title.transAxes)
ax_title.text(0.5, 0.15, 'Decision Tree  ·  Random Forest  ·  Logistic Regression',
              ha='center', va='center', fontsize=13, color='#BDC3C7',
              transform=ax_title.transAxes)

models = [
    {
        'name'    : '🌳  Decision Tree',
        'color'   : '#27AE60',
        'light'   : '#EAFAF1',
        'analogy' : 'Like a flowchart — a series of\nyes/no questions that lead to\na final answer.',
        'when'    : ['✔ Need to explain decision to regulators',
                     '✔ Quick baseline model',
                     '✔ Small/medium datasets',
                     '✔ Non-linear boundaries needed'],
        'pros'    : ['Fully interpretable rules',
                     'Handles mixed data types',
                     'No feature scaling required'],
        'cons'    : ['Prone to overfitting (deep trees)',
                     'Unstable — small data change\ncan give very different tree',
                     'Biased toward high-cardinality\nfeatures'],
        'interp'  : 5,
        'speed'   : 5,
        'accuracy': 3,
    },
    {
        'name'    : '🌲🌲  Random Forest',
        'color'   : '#2980B9',
        'light'   : '#EBF5FB',
        'analogy' : 'Like asking a crowd of experts —\nmany trees vote, and the\nmajority wins.',
        'when'    : ['✔ Best accuracy is the goal',
                     '✔ Noisy / messy real-world data',
                     '✔ Need feature importance',
                     '✔ Tabular data of any size'],
        'pros'    : ['High accuracy out of the box',
                     'Robust to outliers & noise',
                     'Built-in feature importance'],
        'cons'    : ['Hard to interpret individual\ndecisions (black box)',
                     'Slower to train (many trees)',
                     'Memory-intensive for very\nlarge datasets'],
        'interp'  : 2,
        'speed'   : 3,
        'accuracy': 5,
    },
    {
        'name'    : '📈  Logistic Regression',
        'color'   : '#8E44AD',
        'light'   : '#F5EEF8',
        'analogy' : 'Like a scorecard — each feature\ngets a weight, combine them\nto get a probability.',
        'when'    : ['✔ Need probability scores',
                     '✔ Linear relationship expected',
                     '✔ Very fast training needed',
                     '✔ Baseline or credit scoring'],
        'pros'    : ['Extremely fast to train',
                     'Outputs calibrated probabilities',
                     'Works well on high-dim data\n(text, sparse features)'],
        'cons'    : ['Assumes linear decision boundary',
                     'Needs feature scaling',
                     'Struggles with complex\nnon-linear patterns'],
        'interp'  : 4,
        'speed'   : 5,
        'accuracy': 3,
    }
]

col_w  = 1.0 / 3
row_starts = [0.02, 0.60, 0.44, 0.30, 0.14]

def draw_stars(ax, x, y, filled, total=5, color='gold'):
    for i in range(total):
        c = color if i < filled else '#DDDDDD'
        ax.text(x + i*0.04, y, '★', fontsize=12, color=c, transform=ax.transAxes)

for col, m in enumerate(models):
    left = col * col_w

    # Header card
    ax_h = fig.add_axes([left+0.01, 0.78, col_w-0.02, 0.09])
    ax_h.set_facecolor(m['color']); ax_h.axis('off')
    ax_h.text(0.5, 0.65, m['name'], ha='center', va='center',
              fontsize=14, fontweight='bold', color='white', transform=ax_h.transAxes)
    ax_h.text(0.5, 0.20, m['analogy'], ha='center', va='center',
              fontsize=9, color='white', transform=ax_h.transAxes)

    # Ratings
    ax_r = fig.add_axes([left+0.01, 0.68, col_w-0.02, 0.09])
    ax_r.set_facecolor(m['light']); ax_r.axis('off')
    ax_r.text(0.05, 0.85, 'Interpretability', fontsize=9, fontweight='bold', transform=ax_r.transAxes)
    draw_stars(ax_r, 0.55, 0.80, m['interp'])
    ax_r.text(0.05, 0.50, 'Speed', fontsize=9, fontweight='bold', transform=ax_r.transAxes)
    draw_stars(ax_r, 0.55, 0.45, m['speed'])
    ax_r.text(0.05, 0.15, 'Accuracy', fontsize=9, fontweight='bold', transform=ax_r.transAxes)
    draw_stars(ax_r, 0.55, 0.10, m['accuracy'])

    # When to use
    ax_w = fig.add_axes([left+0.01, 0.46, col_w-0.02, 0.21])
    ax_w.set_facecolor('white'); ax_w.axis('off')
    ax_w.text(0.05, 0.94, 'WHEN TO USE', fontsize=10, fontweight='bold',
              color=m['color'], transform=ax_w.transAxes)
    for i, txt in enumerate(m['when']):
        ax_w.text(0.05, 0.76 - i*0.18, txt, fontsize=9, transform=ax_w.transAxes)

    # Pros
    ax_p = fig.add_axes([left+0.01, 0.24, col_w-0.02, 0.21])
    ax_p.set_facecolor('#F9FBE7'); ax_p.axis('off')
    ax_p.text(0.05, 0.92, '✅  PROS', fontsize=10, fontweight='bold',
              color='#27AE60', transform=ax_p.transAxes)
    for i, txt in enumerate(m['pros']):
        ax_p.text(0.05, 0.72 - i*0.22, f'• {txt}', fontsize=9, transform=ax_p.transAxes)

    # Cons
    ax_c = fig.add_axes([left+0.01, 0.03, col_w-0.02, 0.20])
    ax_c.set_facecolor('#FEF9E7'); ax_c.axis('off')
    ax_c.text(0.05, 0.92, '⚠️  CONS', fontsize=10, fontweight='bold',
              color='#E74C3C', transform=ax_c.transAxes)
    for i, txt in enumerate(m['cons']):
        ax_c.text(0.05, 0.70 - i*0.25, f'• {txt}', fontsize=9, transform=ax_c.transAxes)

plt.savefig('model_infographic.png', dpi=150, bbox_inches='tight', facecolor='#F7F9FC')
plt.show()
print("Infographic saved as model_infographic.png")

### D — Evaluation & Critique of AI-Generated Infographic

**What is accurate:**
- The analogy for each model (flowchart / crowd / scorecard) is appropriate for a non-technical audience.
- Relative interpretability ratings (DT > LR > RF) correctly reflect the consensus in ML literature.
- The cons listed (DT overfitting, RF black-box, LR linear boundary) are real and well-documented.

**What could be oversimplified or improved:**
- Accuracy stars are dataset-dependent — RF is rated 5★ here, but on a very linear dataset LR would match it.
- Speed comparison depends heavily on dataset size and `n_estimators` — LR being 5★ speed is true for training but prediction speed for RF is also fast in practice.
- RF interpretability is rated 2★, but with SHAP values it can be made quite explainable; this nuance is missing.

**Corrections applied:** Added the analogy row, separated When/Pros/Cons panels, and used conditional ratings rather than absolute ones.

---
*Day 32 | AM Session | IIT Gandhinagar — AI-ML & Agentic AI Engineering*